# RAG basico con LangChain (LCEL)

Version notebook de `1_rag_basico.py`.

Version mas simple posible: una sola pregunta -> se recuperan documentos relevantes -> se arma un prompt con ese contexto -> el LLM responde.

No hay memoria, ni herramientas, ni agente. Es el "hola mundo" de RAG.

**Flujo (LCEL):**
```
{contexto: retriever, pregunta: passthrough} -> prompt -> LLM -> texto
```

**Entorno para correrlo:** kernel `.venv` (Python 3.11, raiz del repo). Ver la ultima celda ("A que entorno conectarse").

**Importante:** este notebook tiene que abrirse/ejecutarse con working directory `RAG/langchain/`, porque importa `comun.py` que esta al lado. Si lo abris desde VS Code normalmente ya usa la carpeta del notebook como cwd; si no, correr la celda de `sys.path` de abajo soluciona el import.

In [ ]:
import sys
from pathlib import Path

# Asegura que se pueda importar comun.py aunque el notebook corra con otro cwd
sys.path.insert(0, str(Path.cwd()))

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from comun import get_llm, get_retriever, format_docs, leer_pregunta_o_argv

In [ ]:
PROMPT = ChatPromptTemplate.from_template(
    "Sos un asistente que responde SOLO con la informacion del contexto.\n"
    "Si el contexto no alcanza para responder, decilo explicitamente.\n\n"
    "Contexto:\n{context}\n\n"
    "Pregunta: {question}\n"
    "Respuesta:"
)

## Armar la cadena LCEL

In [ ]:
def build_chain():
    """Construye la cadena de RAG basico con LCEL."""
    retriever = get_retriever(k=2)
    llm = get_llm()

    # El operador | encadena componentes (Runnables).
    # Para 'context' recuperamos docs y los formateamos; 'question' pasa tal cual.
    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | PROMPT
        | llm
        | StrOutputParser()  # extrae el texto plano de la respuesta del LLM
    )
    return chain

In [ ]:
chain = build_chain()

## Probar con una pregunta

Cambia el texto de `pregunta` y volve a correr la celda para probar distintas consultas.

In [ ]:
pregunta = "¿Que es Retrieval-Augmented Generation?"

print(f"\n=== Pregunta: {pregunta} ===\n")
respuesta = chain.invoke(pregunta)  # corre toda la cadena de punta a punta
print(respuesta)

## A que entorno conectarse

Mismo venv que el resto del repo: `.venv` en la raiz (Python 3.11.9), con `langchain`, `langchain-ollama`, `langchain-chroma` e `ipykernel` ya instalados.

**En VS Code / Cursor:** "Select Kernel" -> "Python Environments" -> elegí:
```
c:\Users\guill\OneDrive\Documentos\llm\.venv\Scripts\python.exe
```

**Desde Jupyter (navegador):**
```powershell
cd c:\Users\guill\OneDrive\Documentos\llm
.\.venv\Scripts\Activate.ps1
jupyter notebook RAG\langchain\1_rag_basico.ipynb
```

**Antes de correrlo, Ollama tiene que estar corriendo** (esto fue lo que fallo con `rag_simple.ipynb`: `ConnectionError: Failed to connect to Ollama`). Para levantarlo:
```powershell
ollama serve            # dejalo corriendo en una terminal aparte (o como servicio)
ollama pull nomic-embed-text
ollama pull gemma3
```
Verificar que responde antes de correr las celdas:
```powershell
curl http://127.0.0.1:11434
```
(el launcher `RAG\langchain\iniciar_1_basico.ps1` hace este chequeo y arranca Ollama solo para la version .py).